# **Staging Data Ingestion**

## **Overview**

This notebook is responsible for ingesting the **cleaned and validated Olist e-commerce datasets** into the **STAGING layer** of the PostgreSQL database.

The STAGING layer serves as an **intermediate database layer** between the RAW source data and the final analytical structures. Unlike the RAW layer, which preserves the original source datasets, the STAGING layer contains datasets that have already undergone **data cleaning, standardisation, and validation** during the Python data-preparation process.

The cleaned datasets will be loaded from their **exported CSV files** into PostgreSQL using **Python, Pandas, and SQLAlchemy**. Pandas will be used to read the validated datasets, while SQLAlchemy will provide the connection between the Python environment and the PostgreSQL database.

## **Objectives**

This notebook will:

* Establish a connection between Python and the PostgreSQL `ecommerce` database.

* Load the **cleaned and validated datasets** exported from the Python data-preparation process into Pandas DataFrames.

* Ingest the cleaned datasets into the PostgreSQL **`staging` schema**.

* Automatically create the corresponding **staging tables** using Pandas and SQLAlchemy.

* Validate the ingestion by comparing the **cleaned dataset record counts** with the records loaded into the staging tables.

* Confirm that the **staging tables can be successfully queried** from PostgreSQL.

## **Data Layer**

The staging ingestion process follows the next stage of the project architecture:

**Cleaned Olist datasets → Python/Pandas → SQLAlchemy → PostgreSQL STAGING layer**

The STAGING layer contains datasets that have already been **cleaned, standardised, and validated**, but have not yet been transformed into the final analytical model.

Further **database-level transformations, relationships, business rules, analytical structures, and derived metrics** will be applied in subsequent stages of the project.

This notebook focuses specifically on **ingesting the cleaned datasets into the STAGING layer** and does not perform additional **data cleaning, business transformations, or analytical modelling**.

### **1. Import The Libraries**

In [1]:
import pandas as pd

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from getpass import getpass

### **2. Create the SQLAlchemy connection to the Database**

In [2]:
password = getpass("Enter PostgrSQL password:")

In [3]:
connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username="postgres",
    password=password,
    host="localhost",
    port=5432,
    database="ecommerce"
)

engine = create_engine(connection_url)

In [4]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT 1"))
    print(result.scalar())

1


### **3. Load The Datasets**

In [11]:
customers_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_database\01_datasets\cleaned\olist_customers_cleaned.csv")
geolocation_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_database\01_datasets\cleaned\olist_geolocation_cleaned.csv")
items_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_database\01_datasets\cleaned\olist_order_items_cleaned.csv")
payments_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_database\01_datasets\cleaned\olist_order_payments_cleaned.csv")
reviews_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_database\01_datasets\cleaned\olist_order_reviews_cleaned.csv")
orders_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_database\01_datasets\cleaned\olist_orders_cleaned.csv")
products_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_database\01_datasets\cleaned\olist_products_cleaned.csv")
sellers_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_database\01_datasets\cleaned\olist_sellers_cleaned.csv")
translation_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_database\01_datasets\cleaned\olist_category_cleaned.csv")

### **4. Load into the Staging Schema Tables**

#### **Customers Dataset**

In [12]:
# Inspect the dataset
customers_df.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [13]:
# Check the number of rows and columns
customers_df.shape

(99441, 5)

In [14]:
# Load into the staging schema
customers_df.to_sql(
    name="customers",
    con=engine,
    schema="staging",
    if_exists="replace",
    index=False
)

441

In [15]:
# Validate the ingestion
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM staging.customers")
    )
    print(result.scalar())

99441


In [ ]:
# Check the actual table from python
pd.read_sql(
    "SELECT * FROM staging.customers LIMIT 5;",
    engine
)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


#### **Geolocation Dataset**

In [17]:
# Inspect the dataset
geolocation_df.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [18]:
# Check the number of rows and columns
geolocation_df.shape

(720457, 5)

In [20]:
# Load into the staging schema
geolocation_df.to_sql(
    name="geolocation",
    con=engine,
    schema="staging",
    if_exists="replace",
    index=False
)

457

In [21]:
# Validate the ingestion
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM staging.geolocation")
    )
    print(result.scalar())

720457


In [ ]:
# Check the actual table from python
pd.read_sql(
    "SELECT * FROM staging.geolocation LIMIT 5;",
    engine
)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


#### **Order Items Dataset**

In [23]:
# Inspect the dataset
items_df.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [24]:
# Check the number of rows and columns
items_df.shape

(112650, 7)

In [25]:
# Load into the staging schema
items_df.to_sql(
    name="items",
    con=engine,
    schema="staging",
    if_exists="replace",
    index=False
)

650

In [26]:
# Validate the ingestion
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM staging.items")
    )
    print(result.scalar())

112650


In [27]:
# Check the actual table from python
pd.read_sql(
    "SELECT * FROM staging.items LIMIT 5;",
    engine
)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


#### **Order Payments Dataset**

In [28]:
# Inspect the dataset
payments_df.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [29]:
# Check the number of rows and columns
payments_df.shape

(103886, 5)

In [30]:
# Load into the staging schema
payments_df.to_sql(
    name="payments",
    con=engine,
    schema="staging",
    if_exists="replace",
    index=False
)

886

In [31]:
# Validate the ingestion
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM staging.payments")
    )
    print(result.scalar())

103886


In [32]:
# Check the actual table from python
pd.read_sql(
    "SELECT * FROM staging.payments LIMIT 5;",
    engine
)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


#### **Order Reviews Dataset**

In [33]:
# Inspect the dataset
reviews_df.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [34]:
# Check the number of rows and columns
reviews_df.shape

(99224, 7)

In [35]:
# Load into the staging schema
reviews_df.to_sql(
    name="reviews",
    con=engine,
    schema="staging",
    if_exists="replace",
    index=False
)

224

In [36]:
# Validate the ingestion
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM staging.reviews")
    )
    print(result.scalar())

99224


In [37]:
# Check the actual table from python
pd.read_sql(
    "SELECT * FROM staging.reviews LIMIT 5;",
    engine
)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,None,None,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,None,None,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,None,None,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,None,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,None,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


#### **Orders Dataset**

In [38]:
# Inspect the dataset
orders_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [39]:
# Check the number of rows and columns
orders_df.shape

(99441, 8)

In [40]:
# Load in the staging schema
orders_df.to_sql(
    name="orders",
    con=engine,
    schema="staging",
    if_exists="replace",
    index=False
)

441

In [41]:
# Validate the ingestion
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM staging.orders")
    )
    print(result.scalar())

99441


In [42]:
# Check the actual table from python
pd.read_sql(
    "SELECT * FROM staging.orders LIMIT 5;",
    engine
)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


#### **Products Dataset**

In [43]:
# Inspect the dataset
products_df.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [44]:
# Check the number of rows and columns
products_df.shape

(32951, 9)

In [45]:
# Load into the staging schema
products_df.to_sql(
    name="products",
    con=engine,
    schema="staging",
    if_exists="replace",
    index=False
)

951

In [46]:
# Validate the ingestion
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM staging.products")
    )
    print(result.scalar())

32951


In [47]:
# Check the actual table from python
pd.read_sql(
    "SELECT * FROM staging.products LIMIT 5;",
    engine
)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


#### **Sellers Dataset**

In [48]:
# Inspect the dataset
sellers_df.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [49]:
# Check the number of rows and columns
sellers_df.shape

(3095, 4)

In [51]:
# Load into the staging schema
sellers_df.to_sql(
    name="sellers",
    con=engine,
    schema="staging",
    if_exists="replace",
    index=False
)

95

In [52]:
# Validate the ingestion
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM staging.sellers")
    )
    print(result.scalar())

3095


In [53]:
# Check the actual table from python
pd.read_sql(
    "SELECT * FROM staging.sellers LIMIT 5;",
    engine
)

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


#### **Product Name Translation Dataset**

In [54]:
# Inspect the dataset
translation_df.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [55]:
# Check the number of rows and columns
translation_df.shape

(71, 2)

In [56]:
# Load into the staging schema
translation_df.to_sql(
    name="translation",
    con=engine,
    schema="staging",
    if_exists="replace",
    index=False
)

71

In [57]:
# Validate the ingestion
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM staging.translation")
    )
    print(result.scalar())

71


In [58]:
# Check the actual table from python
pd.read_sql(
    "SELECT * FROM staging.translation LIMIT 5;",
    engine
)

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor
